# Preparación de datos — pipeline completo

Notebook único de preparación de datos (fase *Data Preparation* de CRISP-DM). Produce los
conjuntos finales que consumen los cuatro notebooks de modelado.

## Flujo

1. Unificación de focos de calor (NASA FIRMS) y puntos de no incendio
2. Filtro geográfico a la región San Martín y balanceo de clases
3. Variables geográficas y temporales derivadas
4. Fusión con MODIS: temperatura de superficie (MOD11A1) e índices de vegetación (MOD13Q1)
5. Integración meteorológica desde NASA POWER
6. Generación de los dos conjuntos finales

## Salidas

| Archivo | Registros aprox. | Variables | Descripción |
|---|---|---|---|
| `dataset_rutaA_completo.csv` | ~15,000 | 19 | MODIS + POWER. Todas las variables medidas, muestra reducida por nubosidad |
| `dataset_rutaB_power_amplio.csv` | ~86,900 | 16 | Solo POWER. Sin variables ópticas, muestra completa |

## Correcciones aplicadas respecto a la versión anterior

Se corrigieron tres errores que invalidaban las variables climáticas:

1. **Conversión de temperatura.** AppEEARS exporta `LST_Day_1km` ya en Kelvin. La versión
   anterior aplicaba `(valor * 0.02) - 273.15`, que es el factor de escala del *digital
   number* crudo de MODIS, produciendo temperaturas de ~-267 °C. Lo correcto es
   `Kelvin - 273.15`.
2. **Valores de "sin dato" usados como observación.** `LST_Day_1km == 0` indica ausencia de
   observación por nubosidad, no 0 K; `NDVI/EVI == -3000` es el valor de relleno del
   producto. Ambos se descartan antes de convertir.
3. **Imputación con la media global.** La versión anterior rellenaba los nulos del merge con
   la media de una columna ya corrupta, contaminando más de la mitad del conjunto con un
   valor idéntico. Ahora, si no hay observación real, la fila se descarta.

Además se incorpora **EVI**, que ya estaba presente en los archivos MOD13Q1 descargados pero
no se estaba utilizando.


In [1]:
import pandas as pd
import numpy as np
import glob
import json
import os
import time
import requests
from pathlib import Path

pd.set_option('display.width', 140)


In [2]:
# --- Rutas (relativas a la raiz del repositorio) ---
RUTA_INCENDIOS = 'Incendios/*.csv'
RUTA_NO_INCENDIOS = 'NoIncendios/*.csv'
RUTA_MOD11A1 = 'MOD11A1/*.csv'
RUTA_MOD13Q1 = 'MOD13Q1/*.csv'

# --- Area de interes: region San Martin ---
LAT_MIN, LAT_MAX = -8.8, -5.3
LON_MIN, LON_MAX = -77.8, -75.5

# --- Rango temporal del estudio ---
ANIO_INICIO, ANIO_FIN = 2000, 2024

# --- Fusion MODIS ---
TOLERANCIA_MODIS = pd.Timedelta('45 days')  # ventana para emparejar foco con observacion

# --- NASA POWER ---
PASO_LAT, PASO_LON = 0.5, 0.625  # resolucion nativa de MERRA-2
ANIOS_POR_BLOQUE = 5
DIR_CACHE = Path('power_cache');
DIR_CACHE.mkdir(exist_ok=True)
URL_POWER = 'https://power.larc.nasa.gov/api/temporal/daily/point'
# El endpoint 'point' admite hasta 20 parametros por peticion.
# El endpoint 'regional' esta limitado a UNO, por eso no se usa aqui.
PARAMETROS_POWER = ['T2M_MAX', 'T2M_MIN', 'RH2M', 'PRECTOTCORR', 'WS2M', 'WS10M']

# --- Ventanas de agregacion meteorologica ---
VENTANA_LARGA = 30  # dias
VENTANA_CORTA = 7  # dias
UMBRAL_LLUVIA = 1.0  # mm/dia

SEMILLA = 42
print(f"Periodo: {ANIO_INICIO}-{ANIO_FIN} ({ANIO_FIN - ANIO_INICIO + 1} anios)")


Periodo: 2000-2024 (25 anios)


## 1. Unificación de focos de calor y puntos de no incendio

Clase 1: detecciones confirmadas por MODIS (NASA FIRMS).
Clase 0: puntos generados en Google Earth Engine con zona de exclusión de 1 km respecto a
los positivos, y fecha aleatoria dentro del mismo periodo.


In [3]:
# --- Clase 1: incendios (FIRMS) ---
lista_inc = []
for archivo in sorted(glob.glob(RUTA_INCENDIOS)):
    d = pd.read_csv(archivo)
    d = d[['latitude', 'longitude', 'acq_date']].copy()
    d['target'] = 1
    lista_inc.append(d)

# --- Clase 0: no incendios (GEE, coordenadas en la columna .geo como GeoJSON) ---
lista_no = []
for archivo in sorted(glob.glob(RUTA_NO_INCENDIOS)):
    d = pd.read_csv(archivo)
    # En GeoJSON el orden es [longitud, latitud]
    d['longitude'] = d['.geo'].apply(lambda x: json.loads(x)['coordinates'][0])
    d['latitude'] = d['.geo'].apply(lambda x: json.loads(x)['coordinates'][1])
    d = d[['latitude', 'longitude', 'acq_date']].copy()
    d['target'] = 0
    lista_no.append(d)

df_master = pd.concat(lista_inc + lista_no, ignore_index=True)
print(f"Registros unificados: {len(df_master):,}")
print(df_master['target'].value_counts())


Registros unificados: 395,403
target
1    350089
0     45314
Name: count, dtype: int64


In [4]:
# Variables temporales
df_master['Date'] = pd.to_datetime(df_master['acq_date'])
df_master['year'] = df_master['Date'].dt.year
df_master['month'] = df_master['Date'].dt.month
df_master['day'] = df_master['Date'].dt.day
df_master['day_of_year'] = df_master['Date'].dt.dayofyear
df_master = df_master.drop(columns=['acq_date'])

# Eliminacion de duplicados por coordenada y fecha (detecciones satelitales repetidas
# del mismo foco de calor). Descrito en la metodologia, Sec. 3.3.
antes = len(df_master)
df_master = df_master.drop_duplicates(subset=['latitude', 'longitude', 'Date', 'target'])
print(f"Duplicados eliminados: {antes - len(df_master):,}")
print(f"Registros tras deduplicacion: {len(df_master):,}")


Duplicados eliminados: 0
Registros tras deduplicacion: 395,403


## 2. Filtro geográfico y balanceo de clases

In [5]:
df_sm = df_master[
    df_master['latitude'].between(LAT_MIN, LAT_MAX) &
    df_master['longitude'].between(LON_MIN, LON_MAX)
    ].copy()
print(f"Registros en San Martin: {len(df_sm):,}")
print(df_sm['target'].value_counts())

# Balanceo por submuestreo de la clase mayoritaria.
# Se construye explicitamente para no depender del comportamiento de
# groupby().apply(), que cambia entre versiones de pandas.
n_min = df_sm['target'].value_counts().min()
partes_bal = [g.sample(n=n_min, random_state=SEMILLA)
              for _, g in df_sm.groupby('target')]
df_final = (pd.concat(partes_bal)
            .sample(frac=1, random_state=SEMILLA)
            .reset_index(drop=True))

print(f"\nConjunto balanceado: {len(df_final):,} registros")
print(df_final['target'].value_counts())


Registros en San Martin: 120,597
target
1    77154
0    43443
Name: count, dtype: int64

Conjunto balanceado: 86,886 registros
target
1    43443
0    43443
Name: count, dtype: int64


## 3. Variables geográficas y temporales derivadas

In [6]:
# Distancia al ecuador (~111 km por grado de latitud)
df_final['Distance_to_equator'] = df_final['latitude'].abs() * 111

# Epoca seca en San Martin: junio a septiembre
df_final['Is_dry_season'] = df_final['month'].isin([6, 7, 8, 9]).astype(int)

# Claves de union
df_final['lat_round'] = df_final['latitude'].round(2)
df_final['lon_round'] = df_final['longitude'].round(2)
df_final['pow_lat'] = (df_final['latitude'] / PASO_LAT).round() * PASO_LAT
df_final['pow_lon'] = (df_final['longitude'] / PASO_LON).round() * PASO_LON

print(df_final[['Distance_to_equator', 'Is_dry_season']].describe().round(2))


       Distance_to_equator  Is_dry_season
count             86886.00       86886.00
mean                742.25           0.57
std                  87.58           0.50
min                 588.33           0.00
25%                 670.40           0.00
50%                 742.99           1.00
75%                 803.45           1.00
max                 976.79           1.00


## 4. Fusión con MODIS (MOD11A1 y MOD13Q1)

Temperatura de superficie e índices de vegetación NDVI y EVI. Ambos índices provienen del
mismo producto MOD13Q1 ya descargado.


In [7]:
# --- MOD11A1: temperatura de superficie diurna ---
lista_t = []
for f in sorted(glob.glob(RUTA_MOD11A1)):
    lista_t.append(pd.read_csv(f, usecols=['Latitude', 'Longitude', 'Date',
                                           'MOD11A1_061_LST_Day_1km']))
df_temp = pd.concat(lista_t, ignore_index=True)

crudo = len(df_temp)
df_temp = df_temp[df_temp['MOD11A1_061_LST_Day_1km'] > 0]  # 0 = sin observacion valida
df_temp['Temp_max'] = df_temp['MOD11A1_061_LST_Day_1km'] - 273.15  # AppEEARS da Kelvin

df_temp['Date'] = pd.to_datetime(df_temp['Date'])
df_temp['lat_round'] = df_temp['Latitude'].round(2)
df_temp['lon_round'] = df_temp['Longitude'].round(2)
df_temp = df_temp.sort_values('Date')[['Date', 'lat_round', 'lon_round', 'Temp_max']]

print(f"MOD11A1 -> {crudo:,} filas crudas · {len(df_temp):,} con observacion valida "
      f"({len(df_temp) / crudo * 100:.1f}%)")
print(df_temp['Temp_max'].describe().round(2))


MOD11A1 -> 8,984,000 filas crudas · 1,648,983 con observacion valida (18.4%)
count    1648983.00
mean          26.59
std            3.97
min           -3.55
25%           24.57
50%           26.91
75%           29.13
max           47.37
Name: Temp_max, dtype: float64


In [8]:
# --- MOD13Q1: indices de vegetacion NDVI y EVI ---
lista_v = []
for f in sorted(glob.glob(RUTA_MOD13Q1)):
    lista_v.append(pd.read_csv(f, usecols=['Latitude', 'Longitude', 'Date',
                                           'MOD13Q1_061__250m_16_days_NDVI',
                                           'MOD13Q1_061__250m_16_days_EVI']))
df_veg = pd.concat(lista_v, ignore_index=True).rename(columns={
    'MOD13Q1_061__250m_16_days_NDVI': 'NDVI',
    'MOD13Q1_061__250m_16_days_EVI': 'EVI'})

crudo = len(df_veg)
# Descarta el valor de relleno (-3000) y cualquier valor fuera del rango fisico
df_veg = df_veg[df_veg['NDVI'].between(-1, 1) & df_veg['EVI'].between(-1, 1)]

df_veg['Date'] = pd.to_datetime(df_veg['Date'])
df_veg['lat_round'] = df_veg['Latitude'].round(2)
df_veg['lon_round'] = df_veg['Longitude'].round(2)
df_veg = df_veg.sort_values('Date')[['Date', 'lat_round', 'lon_round', 'NDVI', 'EVI']]

print(f"MOD13Q1 -> {crudo:,} filas crudas · {len(df_veg):,} validas "
      f"({len(df_veg) / crudo * 100:.1f}%)")
print(f"Correlacion NDVI-EVI: {df_veg[['NDVI', 'EVI']].corr().iloc[0, 1]:.4f}")


MOD13Q1 -> 621,000 filas crudas · 620,947 validas (100.0%)
Correlacion NDVI-EVI: 0.8136


In [9]:
# Union por coordenada redondeada y fecha mas cercana dentro de la tolerancia.
# Si no hay observacion real, la fila queda nula (NO se imputa con la media).
df_final = df_final.sort_values('Date')
df_final = pd.merge_asof(df_final, df_temp, on='Date', by=['lat_round', 'lon_round'],
                         direction='nearest', tolerance=TOLERANCIA_MODIS)
df_final = pd.merge_asof(df_final, df_veg, on='Date', by=['lat_round', 'lon_round'],
                         direction='nearest', tolerance=TOLERANCIA_MODIS)

cobertura_modis = df_final[['Temp_max', 'NDVI', 'EVI']].notna().all(axis=1).mean()
print(f"Cobertura MODIS tras la fusion: {cobertura_modis * 100:.2f}%")
print(f"  ({df_final[['Temp_max', 'NDVI', 'EVI']].notna().all(axis=1).sum():,} de "
      f"{len(df_final):,} registros)")
print("\nLa perdida se debe a la nubosidad persistente en la Amazonia,")
print("que limita la disponibilidad de observaciones opticas MODIS.")


Cobertura MODIS tras la fusion: 17.45%
  (15,164 de 86,886 registros)

La perdida se debe a la nubosidad persistente en la Amazonia,
que limita la disponibilidad de observaciones opticas MODIS.


## 5. Integración meteorológica — NASA POWER

Aporta humedad, precipitación, viento y días sin lluvia, que MODIS no puede proveer.

**Ventaja decisiva:** POWER es un producto de reanálisis (MERRA-2), no una observación
óptica, por lo que **no tiene huecos por nubosidad**. Cobertura diaria continua desde
1981-01-01, lo que abarca el periodo 2000–2024 con holgura.

**Limitación a declarar:** resolución nativa de 0.5° × 0.625° (~50 km). El área de estudio
abarca solo unas decenas de celdas, por lo que estas variables capturan sobre todo variación
temporal y poca variación espacial intrarregional.

**Estrategia:** en lugar de consultar decenas de miles de puntos, se ajustan las coordenadas
a la malla nativa de POWER y se consulta cada celda única una sola vez por bloque de años.
Las respuestas quedan en caché en `power_cache/`: una re-ejecución no vuelve a descargar.


In [10]:
celdas = (df_final[['pow_lat', 'pow_lon']].drop_duplicates()
          .sort_values(['pow_lat', 'pow_lon']).reset_index(drop=True))
n_bloques = len(range(ANIO_INICIO, ANIO_FIN + 1, ANIOS_POR_BLOQUE))
print(f"Celdas unicas de la malla POWER: {len(celdas)}")
print(f"Peticiones: {len(celdas)} x {n_bloques} bloques = {len(celdas) * n_bloques}")


Celdas unicas de la malla POWER: 32
Peticiones: 32 x 5 bloques = 160


In [11]:
def descargar_celda(lat, lon, anio_ini, anio_fin, reintentos=3):
    """Descarga una celda-bloque de NASA POWER, con cache en disco."""
    ruta = DIR_CACHE / f"power_{lat:.4f}_{lon:.4f}_{anio_ini}_{anio_fin}.json"
    if ruta.exists():
        with open(ruta, 'r') as fh:
            return json.load(fh)

    params = {
        'parameters': ','.join(PARAMETROS_POWER),
        'community': 'AG',
        'longitude': lon, 'latitude': lat,
        'start': f'{anio_ini}0101', 'end': f'{anio_fin}1231',
        'format': 'JSON',
    }
    for intento in range(reintentos):
        try:
            r = requests.get(URL_POWER, params=params, timeout=120)
            r.raise_for_status()
            datos = r.json()
            with open(ruta, 'w') as fh:
                json.dump(datos, fh)
            return datos
        except Exception as e:
            if intento == reintentos - 1:
                print(f"  FALLO {lat},{lon} {anio_ini}-{anio_fin}: {e}")
                return None
            time.sleep(3 * (intento + 1))
    return None


In [12]:
bloques, fallos = [], []
t0 = time.time()

for i, fila in celdas.iterrows():
    lat, lon = fila['pow_lat'], fila['pow_lon']
    for anio_ini in range(ANIO_INICIO, ANIO_FIN + 1, ANIOS_POR_BLOQUE):
        anio_fin = min(anio_ini + ANIOS_POR_BLOQUE - 1, ANIO_FIN)
        datos = descargar_celda(lat, lon, anio_ini, anio_fin)

        if datos is None:
            fallos.append((lat, lon, anio_ini, anio_fin));
            continue
        try:
            series = datos['properties']['parameter']
        except (KeyError, TypeError):
            fallos.append((lat, lon, anio_ini, anio_fin));
            continue

        recibidos = [p for p in PARAMETROS_POWER if p in series]
        if not recibidos:
            fallos.append((lat, lon, anio_ini, anio_fin));
            continue

        fechas = sorted(series[recibidos[0]].keys())
        d = {'Date': fechas}
        for p in PARAMETROS_POWER:
            d[p] = [series.get(p, {}).get(f, np.nan) for f in fechas]
        b = pd.DataFrame(d)
        b['pow_lat'], b['pow_lon'] = lat, lon
        bloques.append(b)

    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{len(celdas)} celdas · {time.time() - t0:.0f}s")

print(f"\nDescarga completada en {time.time() - t0:.0f}s")
print(f"Bloques obtenidos: {len(bloques)} · Fallos: {len(fallos)}")
if fallos:
    print("Re-ejecute esta celda para reintentar los bloques fallidos.")


  5/32 celdas · 41s
  10/32 celdas · 79s
  15/32 celdas · 118s
  20/32 celdas · 161s
  25/32 celdas · 198s
  30/32 celdas · 238s

Descarga completada en 253s
Bloques obtenidos: 160 · Fallos: 0


In [13]:
clima = pd.concat(bloques, ignore_index=True)
clima['Date'] = pd.to_datetime(clima['Date'], format='%Y%m%d')
clima = clima.replace(-999, np.nan)  # -999 es el valor de relleno de POWER

recibidos = [p for p in PARAMETROS_POWER if p in clima.columns]
print(f"Filas: {len(clima):,} · Fechas: {clima['Date'].min().date()} a "
      f"{clima['Date'].max().date()} · Anios: {clima['Date'].dt.year.nunique()}")

# Comprobaciones de plausibilidad fisica para selva tropical
rangos = {'T2M_MAX': (5, 50, 'C'), 'T2M_MIN': (0, 40, 'C'), 'RH2M': (0, 100, '%'),
          'PRECTOTCORR': (0, 500, 'mm/dia'), 'WS2M': (0, 30, 'm/s'),
          'WS10M': (0, 40, 'm/s')}
print()
for p in recibidos:
    lo, hi, uni = rangos[p]
    prop = clima[p].between(lo, hi).mean()
    print(f"[{'OK     ' if prop > 0.99 else 'REVISAR'}] {p:12s} en [{lo}, {hi}] "
          f"{uni:8s} -> {prop * 100:6.2f}%")

if 'RH2M' in recibidos and clima['RH2M'].median() <= 1.0:
    print("\nRH2M parece venir en fraccion (0-1). Ejecute: clima['RH2M'] *= 100")


Filas: 292,224 · Fechas: 2000-01-01 a 2024-12-31 · Anios: 25

[OK     ] T2M_MAX      en [5, 50] C        -> 100.00%
[OK     ] T2M_MIN      en [0, 40] C        ->  99.74%
[OK     ] RH2M         en [0, 100] %        -> 100.00%
[OK     ] PRECTOTCORR  en [0, 500] mm/dia   -> 100.00%
[OK     ] WS2M         en [0, 30] m/s      -> 100.00%
[OK     ] WS10M        en [0, 40] m/s      -> 100.00%


### Derivación de las variables meteorológicas

Las ventanas móviles se calculan **por celda** y de forma estrictamente **causal**: con
`closed='left'` la ventana termina el día *anterior* al registro y nunca incluye el día que
se está prediciendo. Esto evita fuga de información (*data leakage*).

| Variable | Derivación |
|---|---|
| `Temp_max_avg` / `Temp_max_max` | media / máximo móvil 30 d de `T2M_MAX` |
| `Humidity_avg` / `Humidity_min` | media / mínimo móvil 30 d de `RH2M` |
| `Precip_total` | suma móvil 30 d de `PRECTOTCORR` |
| `Days_no_rain` | días con `PRECTOTCORR` < 1 mm en los 7 días previos |
| `Wind_avg` / `Wind_max` | media / máximo móvil 30 d de `WS2M` |


In [14]:
partes = []
for (la, lo), g in clima.groupby(['pow_lat', 'pow_lon']):
    g = g.sort_values('Date').set_index('Date').copy()
    vl, vc = f'{VENTANA_LARGA}D', f'{VENTANA_CORTA}D'

    g['Temp_max_avg'] = g['T2M_MAX'].rolling(vl, closed='left', min_periods=1).mean()
    g['Temp_max_max'] = g['T2M_MAX'].rolling(vl, closed='left', min_periods=1).max()
    g['Humidity_avg'] = g['RH2M'].rolling(vl, closed='left', min_periods=1).mean()
    g['Humidity_min'] = g['RH2M'].rolling(vl, closed='left', min_periods=1).min()
    g['Precip_total'] = g['PRECTOTCORR'].rolling(vl, closed='left', min_periods=1).sum()
    g['Wind_avg'] = g['WS2M'].rolling(vl, closed='left', min_periods=1).mean()
    g['Wind_max'] = g['WS2M'].rolling(vl, closed='left', min_periods=1).max()

    seco = (g['PRECTOTCORR'] < UMBRAL_LLUVIA).astype(float)
    g['Days_no_rain'] = seco.rolling(vc, closed='left', min_periods=1).sum()
    partes.append(g.reset_index())

clima_deriv = pd.concat(partes, ignore_index=True)

VARS_METEO = ['Temp_max_avg', 'Temp_max_max', 'Humidity_avg', 'Humidity_min',
              'Precip_total', 'Days_no_rain', 'Wind_avg', 'Wind_max']
print(clima_deriv[VARS_METEO].describe().round(2))


       Temp_max_avg  Temp_max_max  Humidity_avg  Humidity_min  Precip_total  Days_no_rain   Wind_avg   Wind_max
count     292192.00     292192.00     292192.00     292192.00     292192.00     292192.00  292192.00  292192.00
mean          25.97         28.84         77.09         66.58         86.24          4.22       0.63       0.92
std            5.49          5.70          8.62         10.94         67.96          1.89       0.82       1.16
min           12.66         13.29         42.31         17.53          0.00          0.00       0.00       0.00
25%           21.35         24.15         71.69         59.18         36.35          3.00       0.06       0.14
50%           27.27         29.76         77.58         66.62         70.21          4.00       0.11       0.23
75%           30.09         33.36         83.56         74.40        118.62          6.00       1.25       1.68
max           39.01         40.99         93.79         90.70        665.64          7.00       4.27    

In [15]:
# --- Verificacion anti-leakage ---
excede = (clima_deriv['T2M_MAX'] > clima_deriv['Temp_max_max']).sum()
print(f"Dias donde T2M_MAX supera su Temp_max_max de ventana: {excede:,}")
print("(> 0 confirma que la ventana excluye el dia actual: sin data leakage)\n")

la0, lo0 = clima_deriv['pow_lat'].iloc[0], clima_deriv['pow_lon'].iloc[0]
sub = clima_deriv[(clima_deriv['pow_lat'] == la0) &
                  (clima_deriv['pow_lon'] == lo0)].sort_values('Date').head(40)
fila = sub.iloc[35]
prev = sub[(sub['Date'] < fila['Date']) &
           (sub['Date'] >= fila['Date'] - pd.Timedelta(days=VENTANA_LARGA))]
print(f"Comprobacion manual para {fila['Date'].date()}:")
print(f"  Temp_max_avg calculado       : {fila['Temp_max_avg']:.6f}")
print(f"  Media de los 30 dias previos : {prev['T2M_MAX'].mean():.6f}")
print(f"  Coinciden: {abs(fila['Temp_max_avg'] - prev['T2M_MAX'].mean()) < 1e-9}")
print(f"\nNulos: {clima_deriv[VARS_METEO].isnull().sum().sum()} "
      f"(esperado {len(celdas) * len(VARS_METEO)}: primer dia de cada celda)")


Dias donde T2M_MAX supera su Temp_max_max de ventana: 16,324
(> 0 confirma que la ventana excluye el dia actual: sin data leakage)

Comprobacion manual para 2000-02-05:
  Temp_max_avg calculado       : 14.717667
  Media de los 30 dias previos : 14.717667
  Coinciden: True

Nulos: 256 (esperado 256: primer dia de cada celda)


In [16]:
# Union exacta por celda y fecha (POWER tiene cobertura diaria continua)
df_final = df_final.merge(
    clima_deriv[['pow_lat', 'pow_lon', 'Date'] + VARS_METEO],
    on=['pow_lat', 'pow_lon', 'Date'], how='left')

cobertura_power = df_final[VARS_METEO].notna().all(axis=1).mean()
print(f"Cobertura POWER : {cobertura_power * 100:6.2f}%  "
      f"({df_final[VARS_METEO].notna().all(axis=1).sum():,} registros)")
print(f"Cobertura MODIS : {cobertura_modis * 100:6.2f}%  "
      f"({df_final[['Temp_max', 'NDVI', 'EVI']].notna().all(axis=1).sum():,} registros)")


Cobertura POWER :  99.99%  (86,880 registros)
Cobertura MODIS :  17.45%  (15,164 registros)


## 6. Generación de los conjuntos finales

Se producen dos conjuntos que reflejan un compromiso real entre cobertura de variables y
tamaño muestral. Se recomienda entrenar con ambos y reportar la comparación.


In [17]:
VARS_BASE = ['latitude', 'longitude', 'month', 'day', 'day_of_year', 'year',
             'Distance_to_equator', 'Is_dry_season']
VARS_MODIS = ['Temp_max', 'NDVI', 'EVI']

COLUMNAS_A = ['target', 'Date'] + VARS_BASE + VARS_MODIS + VARS_METEO
COLUMNAS_B = ['target', 'Date'] + VARS_BASE + VARS_METEO

# Ruta A: completa (MODIS + POWER)
ruta_a = df_final.dropna(subset=VARS_MODIS + VARS_METEO)[COLUMNAS_A].reset_index(drop=True)
ruta_a.to_csv('dataset_rutaA_completo.csv', index=False)

# Ruta B: amplia (solo POWER)
ruta_b = df_final.dropna(subset=VARS_METEO)[COLUMNAS_B].reset_index(drop=True)
ruta_b.to_csv('dataset_rutaB_power_amplio.csv', index=False)

resumen = pd.DataFrame([
    {'Ruta': 'A - completa (MODIS + POWER)', 'Registros': len(ruta_a),
     'Variables': len(VARS_BASE) + len(VARS_MODIS) + len(VARS_METEO),
     'Clase 1 (%)': round(ruta_a['target'].mean() * 100, 1),
     'Archivo': 'dataset_rutaA_completo.csv'},
    {'Ruta': 'B - amplia (solo POWER)', 'Registros': len(ruta_b),
     'Variables': len(VARS_BASE) + len(VARS_METEO),
     'Clase 1 (%)': round(ruta_b['target'].mean() * 100, 1),
     'Archivo': 'dataset_rutaB_power_amplio.csv'},
])
print(resumen.to_string(index=False))
resumen.to_csv('resumen_datasets.csv', index=False)


                        Ruta  Registros  Variables  Clase 1 (%)                        Archivo
A - completa (MODIS + POWER)      15164         19         28.7     dataset_rutaA_completo.csv
     B - amplia (solo POWER)      86880         16         50.0 dataset_rutaB_power_amplio.csv


In [18]:
# Comparacion meteorologica entre clases: material directo para Resultados
comp = ruta_b.groupby('target')[VARS_METEO].mean().T
comp.columns = ['No incendio (0)', 'Incendio (1)']
comp['Diferencia %'] = ((comp['Incendio (1)'] / comp['No incendio (0)'] - 1) * 100).round(1)
print(comp.round(3))
comp.to_csv('comparacion_meteorologica_clases.csv')


              No incendio (0)  Incendio (1)  Diferencia %
Temp_max_avg           28.266        29.349           3.8
Temp_max_max           31.245        32.043           2.6
Humidity_avg           76.291        68.317         -10.5
Humidity_min           65.813        57.331         -12.9
Precip_total           84.300        46.964         -44.3
Days_no_rain            4.200         5.444          29.6
Wind_avg                0.310         0.286          -7.9
Wind_max                0.477         0.437          -8.4
